In [66]:
import numpy as np
import pandas as pd
import scipy.stats as stats

df_list = []
for w in range(4):
    df_raw = pd.read_csv(f'../data/hospitalizations/hospitalizations_{w+1}wk.csv', index_col=0)
    
    df = df_raw[(df_raw.geo_value == 'us')]
    valid_dates = sorted(df.target_end_date[df.actual.notna()].unique())
    df_f = df.loc[:, [c for c in df.columns if c.startswith('forecast_0')]]
    print(f"{w+1}wk data shape: {df.shape}")
    print(f"Number of non NA entries: {df_f.notna().sum().sum()}, Actual exists for {len(valid_dates)} dates")
    print(f"First and last 3 dates with non-NA Y: {valid_dates[:3]} ... {valid_dates[-3:]}")
    print(f"Target date range: {df.target_end_date.min()} to {df.target_end_date.max()}")
    # print(f"Range of ahead: {df.ahead.min()} to {df.ahead.max()}")
    
    df.drop(columns=['geo_value'], inplace=True)
    df_list.append(df)
    

1wk data shape: (19075, 31)
Number of non NA entries: 434602, Actual exists for 1049 dates
First and last 3 dates with non-NA Y: ['2020-07-27', '2020-07-28', '2020-07-29'] ... ['2023-06-08', '2023-06-09', '2023-06-10']
Target date range: 2020-03-28 to 2023-06-12
2wk data shape: (19048, 31)
Number of non NA entries: 433981, Actual exists for 1049 dates
First and last 3 dates with non-NA Y: ['2020-07-27', '2020-07-28', '2020-07-29'] ... ['2023-06-08', '2023-06-09', '2023-06-10']
Target date range: 2020-04-04 to 2023-06-19
3wk data shape: (18433, 31)
Number of non NA entries: 419836, Actual exists for 1049 dates
First and last 3 dates with non-NA Y: ['2020-07-27', '2020-07-28', '2020-07-29'] ... ['2023-06-08', '2023-06-09', '2023-06-10']
Target date range: 2020-04-11 to 2023-06-26
4wk data shape: (18139, 31)
Number of non NA entries: 413074, Actual exists for 1049 dates
First and last 3 dates with non-NA Y: ['2020-07-27', '2020-07-28', '2020-07-29'] ... ['2023-06-08', '2023-06-09', '2023-

In [ ]:
df_list2 = []
forecasters_stat = {'occurrences': {}, 'min_data_len': {}, 'max_data_len': {}, 'first_date': {}, 'last_date': {}}
n_forecasters_to_take = 20
forecasts_to_take = set()

for w, df in enumerate(df_list):
    df2 = df[(df.target_end_date >= '2020-07-27') & (df.target_end_date <= '2023-06-10')]
    print(f"{w+1}wk data")
    print("# of rows after filtering dates:", df2.shape)   # Actual column has no NA after this

    for f_name, count in zip(*np.unique(df2.forecaster, return_counts=True)):
        forecasters_stat['occurrences'][f_name] = forecasters_stat['occurrences'].get(f_name, 0) + 1
        forecasters_stat['min_data_len'][f_name] = min(forecasters_stat['min_data_len'].get(f_name, float('inf')), count)
        forecasters_stat['max_data_len'][f_name] = max(forecasters_stat['max_data_len'].get(f_name, 0), count)
        forecasters_stat['first_date'][f_name] = min(forecasters_stat['first_date'].get(f_name, '2024-01-01'), df2[df2.forecaster == f_name].target_end_date.min())
        forecasters_stat['last_date'][f_name] = max(forecasters_stat['last_date'].get(f_name, '2020-01-01'), df2[df2.forecaster == f_name].target_end_date.max())
    
    df_list2.append(df2)


print("-" * 100)

sorted_forecasters = sorted(forecasters_stat['occurrences'].keys(), key=lambda k: -forecasters_stat['max_data_len'][k])
for f in sorted_forecasters:
    print(f"{f:<30}, min_len: {forecasters_stat['min_data_len'][f]:>5}, max_len: {forecasters_stat['max_data_len'][f]:>5}, first_date: {forecasters_stat['first_date'][f]:>5}, last_date: {forecasters_stat['last_date'][f]:>5}, occurrences: {forecasters_stat['occurrences'][f]:>5}")
    print(np.unique(pd.to_datetime(df2[df2.forecaster == f].forecast_date.unique(), format="%Y-%m-%d").dayofweek, return_counts=True))
print("-" * 100)

# for idx, df in enumerate(df_list2):
#     df_list2[idx] = df[df.forecaster.isin(forecaster_list)]
#     print(f"{idx+1}wk data")
#     print("# of rows after filtering forecasters:", df_list2[idx].shape)



1wk data
# of rows after filtering dates: (18384, 30)
2wk data
# of rows after filtering dates: (18307, 30)
3wk data
# of rows after filtering dates: (17645, 30)
4wk data
# of rows after filtering dates: (17309, 30)
----------------------------------------------------------------------------------------------------
CU-select                     , min_len:  1043, max_len:  1068, first_date: 2020-07-27, last_date: 2023-04-02, occurrences:     4
(array([3, 6], dtype=int32), array([ 17, 136]))
GT-DeepCOVID                  , min_len:   979, max_len:   988, first_date: 2020-07-27, last_date: 2023-06-10, occurrences:     4
(array([0], dtype=int32), array([142]))
COVIDhub-4_week_ensemble      , min_len:   894, max_len:   915, first_date: 2020-12-08, last_date: 2023-06-10, occurrences:     4
(array([0], dtype=int32), array([128]))
COVIDhub-baseline             , min_len:   894, max_len:   915, first_date: 2020-12-08, last_date: 2023-06-10, occurrences:     4
(array([0], dtype=int32), array([12

In [25]:
sorted_forecasters

['CU-select',
 'GT-DeepCOVID',
 'COVIDhub-4_week_ensemble',
 'COVIDhub-baseline',
 'COVIDhub-ensemble',
 'COVIDhub_CDC-ensemble',
 'Karlen-pypm',
 'JHU_IDD-CovidSP',
 'MOBS-GLEAM_COVID',
 'COVIDhub-trained_ensemble',
 'USC-SI_kJalpha',
 'JHUAPL-Bucky',
 'JHUAPL-SLPHospEns',
 'JHUAPL-Gecko',
 'MUNI-ARIMA',
 'CMU-TimeSeries',
 'UMass-trends_ensemble',
 'PSI-DICE',
 'prolix-euclidean',
 'IHME-CurveFit',
 'CUB_PopCouncil-SLSTM',
 'BPagano-RtDriven',
 'LANL-GrowthRate',
 'UVA-Ensemble',
 'Covid19Sim-Simulator',
 'UMass-sarix',
 'UT-Osiris',
 'UCLA-SuEIR',
 'LUcompUncertLab-VAR_3streams',
 'USACE-ERDC_SEIR',
 'UMass-gbq',
 'CEPH-Rtrend_covid',
 'JHUAPLTDWG-ICATTML',
 'SGroup-RandomForest',
 'JHUAPL-Morris']

### Build `df_data_dic`

In [27]:
# For lin. interp. version,
forecaster_list = ['COVIDhub-4_week_ensemble', 'JHUAPL-SLPHospEns', 'CU-select', 'GT-DeepCOVID', 'COVIDhub-baseline', 'Karlen-pypm',
 'JHU_IDD-CovidSP', 'MOBS-GLEAM_COVID', 'USC-SI_kJalpha', 'JHUAPL-Bucky', 'JHUAPL-Gecko', 'MUNI-ARIMA', 'CMU-TimeSeries',
 'UMass-trends_ensemble', 'PSI-DICE', 'prolix-euclidean', 'IHME-CurveFit', 'CUB_PopCouncil-SLSTM', 'BPagano-RtDriven',
 'LANL-GrowthRate', 'UVA-Ensemble']

# for f_name in forecaster_list:
#     print(f_name)
#     print(f'https://github.com/reichlab/covid19-forecast-hub/blob/master/data-processed/{f_name}/metadata-{f_name}.txt')

# For past Y impute version, 
# forecaster_list = ['CU-select', 'GT-DeepCOVID', 'COVIDhub-baseline', 'Karlen-pypm', 'JHU_IDD-CovidSP', 'MOBS-GLEAM_COVID'] + ['COVIDhub-4_week_ensemble']


# Given df, take the latest forecast
def take_last_among_dup (df):
    assert all(col in df.columns for col in ['target_end_date', 'forecast_date'])
    return df.loc[df.groupby('target_end_date')['forecast_date'].idxmax()]

df_data_dic = {}

for w in range(4):
    print("-" * 100)
    print(f"Processing {w+1}wk data")
    df_data_dic[w+1] = {}

    print('Target date with non-zero std Actual (across different forecasters): ', np.sum(df_list2[w].groupby('target_end_date').actual.std() > 0))

    for f_name in forecaster_list:
        df_here = df_list2[w][df_list2[w].forecaster == f_name]   
        date_start = pd.to_datetime(df_here.target_end_date.min())
        date_end = pd.to_datetime(df_here.target_end_date.max())
        date_range_length = (date_end - date_start).days + 1  # inclusive of endpoints
        
        # Duplicate check 
        ud, rc = np.unique(df_here.target_end_date, return_counts=True)
        n_missing = date_range_length - len(ud)
        print(f'{f_name:>25}: Rows {df_here.shape[0]:>5}, First date: {date_start.date()}, Last date: {date_end.date()}, Missing: {n_missing}')
        missing_dates = pd.date_range(start=date_start, end=date_end).strftime('%Y-%m-%d').difference(df_here.target_end_date)
        if False:
            print(missing_dates)
        if np.sum(rc > 1) >= 1:
            dup_dates = np.sort(ud[rc > 1])
            print(f"{0:>27}Contains {len(dup_dates)} duplicate forecasts from {dup_dates[0]} to {dup_dates[-1]}")
            df_here = take_last_among_dup(df_here)
        
        df_data_dic[w+1][f_name] = df_here.drop(columns=['ahead', 'forecaster', 'forecast_date', 'forecast_NaN', 'actual'])

----------------------------------------------------------------------------------------------------
Processing 1wk data
Target date with non-zero std Actual (across different forecasters):  0
 COVIDhub-4_week_ensemble: Rows   915, First date: 2020-12-08, Last date: 2023-06-10, Missing: 0
        JHUAPL-SLPHospEns: Rows   567, First date: 2020-12-07, Last date: 2022-09-05, Missing: 71
                CU-select: Rows  1043, First date: 2020-07-27, Last date: 2023-03-12, Missing: 28
                          0Contains 112 duplicate forecasts from 2020-08-07 to 2020-12-31
             GT-DeepCOVID: Rows   979, First date: 2020-07-27, Last date: 2023-06-10, Missing: 70
        COVIDhub-baseline: Rows   915, First date: 2020-12-08, Last date: 2023-06-10, Missing: 0
              Karlen-pypm: Rows   875, First date: 2020-07-27, Last date: 2023-02-26, Missing: 70
          JHU_IDD-CovidSP: Rows   825, First date: 2020-07-27, Last date: 2023-06-10, Missing: 226
                          0Conta

## Imputation + Dates to run

### (031226 version) Imputation with most recent Y (bad), Dates: COVIDhub-baseline

In [ ]:
# Imputation
g = df_list2[0].groupby("target_end_date")["actual"]

result = pd.DataFrame({
    "actual": g.first(),                       # candidate value
    "any_na": g.apply(lambda x: x.isna().any()),
    "n_unique_non_na": g.nunique(dropna=True)  # number of unique actual values
})

result["all_equal"] = result["n_unique_non_na"] == 1
result["valid"] = (~result["any_na"]) & (result["all_equal"])
print(f'Start date: {result.index.min()}, End date: {result.index.max()}, # of dates in data: {result.shape[0]}')
print(f'Total dates: {(pd.to_datetime(result.index.max()) - pd.to_datetime(result.index.min())).days + 1}')
print(f'Not valid dates: {result.shape[0] - result["valid"].sum()}')


alpha_list = [float(col[10:]) for col in df_list2[0].columns if col.startswith('forecast_0')]
Y = result['actual'].sort_index()
Y_impute_prev_Y = {}
for w in range(1,5):
    Y_impute_prev_Y[w] = {}
    Y_delayed = Y.shift(7*w).fillna(0)
    Y_7days_std_delayed = Y.rolling(7).std().shift(7*w).fillna(0) 
    for alpha in alpha_list:
        Y_impute_prev_Y[w][alpha] = Y_delayed + Y_7days_std_delayed * stats.norm.ppf(alpha)

Start date: 2020-07-27, End date: 2023-06-10, # of dates in data: 1049
Total dates: 1049
Not valid dates: 0


In [ ]:
df_covidhub_4wk = df_data_dic[4]['COVIDhub-baseline']
start_date = df_covidhub_4wk.target_end_date.min()
end_date = df_covidhub_4wk.target_end_date.max()
dates_list = pd.date_range(start=start_date, end=end_date).strftime('%Y-%m-%d')

forecasts_dict = {}
for w in range(1,5):
    forecasts_dict[w] = {}
    for f_name in forecaster_list:
        forecasts_dict[w][f_name] = {}
        for alpha in alpha_list:
            forecasts_dict[w][f_name][alpha] = (df_data_dic[w][f_name][[f'forecast_{alpha}', 'target_end_date']]
                                                    .set_index('target_end_date')[f'forecast_{alpha}']
                                                    .reindex(dates_list)
                                                    .fillna(Y_impute_prev_Y[w][alpha])
                                                    )

# Weekly aggregate
# df4 = df3.drop(columns=['ahead', 'target_end_date', 'forecast_NaN']).groupby(['forecaster', 'forecast_date'], as_index=False).sum()
# Y = {date: df4.loc[df4['forecast_date'] == date, 'actual'].values[0] for date in sorted(df4['forecast_date'].unique())}

In [68]:
import pickle
d = {}
d['forecasts_dict'] = forecasts_dict
d['alpha_list'] = alpha_list
d['dates_list'] = dates_list
d['forecaster_list'] = forecaster_list
d['Y'] = Y

pickle.dump(d, open('../data/hospitalizations/preprocess_data.pkl', 'wb'))

### (031426 version) Impute with median of others interpolation, Dates: COVIDhub-trained_ensemble

In [29]:
g = df_list2[0].groupby("target_end_date")["actual"]

result = pd.DataFrame({
    "actual": g.first(),                       # candidate value
    "any_na": g.apply(lambda x: x.isna().any()),
    "n_unique_non_na": g.nunique(dropna=True)  # number of unique actual values
})

alpha_list = [float(col[10:]) for col in df_list2[0].columns if col.startswith('forecast_0')]
Y = result['actual'].sort_index()


start_date = df_data_dic[4]['COVIDhub-4_week_ensemble'].target_end_date.min()
end_date = df_data_dic[4]['COVIDhub-4_week_ensemble'].target_end_date.max()
print(start_date, end_date)
dates_list = pd.date_range(start=start_date, end=end_date).strftime('%Y-%m-%d')

forecasts_dict = {}
for w in range(1,5):
    forecasts_dict[w] = {}
    for f_name in forecaster_list:
        forecasts_dict[w][f_name] = {}
        for alpha in alpha_list:
            forecasts_dict[w][f_name][alpha] = (df_data_dic[w][f_name][[f'forecast_{alpha}', 'target_end_date']]
                                                    .set_index('target_end_date')[f'forecast_{alpha}']
                                                    .reindex(dates_list)
                                                    )
            
    # interpolate using avg of others
    for alpha in alpha_list:
        avg_of_others = (forecasts_dict[w][f_name][alpha] for f_name in forecaster_list if f_name not in ['COVIDhub-baseline', 'COVIDhub-trained_ensemble', 'JHUAPL-SLPHospEns'])
        avg_of_others = pd.concat(avg_of_others, axis=1).median(axis=1, skipna=True)
        for f_name in forecaster_list:    
           forecasts_dict[w][f_name][alpha] = forecasts_dict[w][f_name][alpha].fillna(avg_of_others)

2020-12-29 2023-06-10


In [11]:
import pickle
d = {}
d['forecasts_dict'] = forecasts_dict
d['alpha_list'] = alpha_list
d['dates_list'] = dates_list
d['forecaster_list'] = forecaster_list
d['Y'] = Y

pickle.dump(d, open('../data/hospitalizations/preprocess_data_medianinterp.pkl', 'wb'))

## Weekly data
Required to run imputation above first

COVIDhub ensemble is released on Monday, for Tues to next Mon.

In [12]:
assert dates_list.size == (pd.to_datetime(dates_list[-1]) - pd.to_datetime(dates_list[0])).days + 1
print(pd.to_datetime(dates_list[0]).strftime('%A'))  
print(f'Size of dates_list: {dates_list.size}, # of weeks: {dates_list.size // 7}')

dates_list_w = dates_list[:(dates_list.size // 7) * 7]
print(f'Trimmed size: {dates_list_w.size}')

def weekly_agg(series, dates_list_7):
    series = series[dates_list_7]
    group_index = np.arange(len(series)) // 7
    assert len(series) % 7 == 0
    series_w = series.groupby(group_index).sum()
    series_w.index = series.index[::7]
    return series_w

dates_list_7 = dates_list[:(dates_list.size // 7) * 7]
dates_list_w = dates_list_7[::7]

Y_w = weekly_agg(Y, dates_list_7)

forecasts_dict_w = {}
for w in range(1,5):
    forecasts_dict_w[w] = {}
    for f_name in forecaster_list:
        forecasts_dict_w[w][f_name] = {}
        for alpha in alpha_list:
            forecasts_dict_w[w][f_name][alpha] = weekly_agg(forecasts_dict[w][f_name][alpha], dates_list_7)

Tuesday
Size of dates_list: 838, # of weeks: 119
Trimmed size: 833


In [13]:
import pickle
d = {}
d['forecasts_dict'] = forecasts_dict_w
d['alpha_list'] = alpha_list
d['dates_list'] = dates_list_w
d['forecaster_list'] = forecaster_list
d['Y'] = Y_w

pickle.dump(d, open('../data/hospitalizations/preprocess_data_medianinterp_weekly.pkl', 'wb'))